# GUS05 — Estimated Population, Pop-Class Distribution & JSON Export

This notebook demonstrates the v5.2 features added to `geoTERYT_db`:

1. **`est_pop`** — extract estimated total population from E_age_sex cross tables
2. **`classify_est_population()`** — classify gminas (rodz 1/2/3) into population classes
   - rodz 1 (urban): entire pop → city-size class
   - rodz 2 (rural): entire pop → 'wieś'
   - rodz 3 (urban-rural): split via children rodz 4 (town) / rodz 5 (rural area)
3. **`aggregate_pop_class()`** — roll up gmina distributions to powiat & voivodeship
4. **`export_tables_json()`** — export estimated tables as JSON dictionaries

Population class codes:
| Code | Label |
|------|-------|
| 1 | wieś |
| 2 | miasto do 20 000 |
| 3 | miasto od 20 001 do 50 000 |
| 4 | miasto od 50 001 do 100 000 |
| 5 | miasto od 100 001 do 500 000 |
| 6 | miasto 500 001 i więcej |

In [1]:
# ── Cell 1: Setup & load database ──
import sys, os, time
import numpy as np
import pandas as pd
import json
import gc

REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(REPO, 'Code', 'tools'))

DATA_ROOT = os.path.join(REPO, '..', '..', 'Data', 'Geospatial')
JSON_ROOT = os.path.join(REPO, '..', '..', 'Data', 'CT JSON')
DB_INPUT  = os.path.join(DATA_ROOT, 'geoteryt_E.pkl')
DB_OUTPUT = os.path.join(DATA_ROOT, 'geoteryt_E.pkl')  # overwrite with new attrs

from geoTERYT_db import (
    GeoTERYTDatabase, load_complete_database, TERYTRecord,
    LEVEL_GMINA, LEVEL_POWIAT, LEVEL_VOIVODESHIP,
    RODZ_AGGREGATION_SET, YEAR_RANGE_FULL, DATETIME_INDEX_FULL,
)

print(f"Loading database from: {DB_INPUT}")
t0 = time.time()
db = load_complete_database(DB_INPUT, verbose=True)
elapsed = time.time() - t0
print(f"\nLoaded in {elapsed:.1f}s — {len(db._records)} records")

gc.collect()

Loading database from: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/../../Data/Geospatial/geoteryt_E.pkl
Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/../../Data/Geospatial/geoteryt_E.pkl...
  Database version: 4.3
  ✓ Restored old voivodships: 49 rows
  ✓ Restored geometry store: 13,287 unique geometries
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4613 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3662
  ✓ Records with old_woj: 4104
  ✓ Records with data: 4592
  ✓ Records with cross tables: 4592
  ✓ Records with population data: 4590
  ✓ Records with pop_class: 4542

Loaded in 351.2s — 4613 records


0

## Step 1: Extract Estimated Population (`est_pop`)

In [2]:
# ── Cell 2: Extract est_pop from E_age_sex cross tables ──
t0 = time.time()
n_extracted = db.extract_est_population(verbose=True)
elapsed = time.time() - t0
print(f"\nCompleted in {elapsed:.1f}s")

# Summary statistics
gminas = [r for r in db._records.values()
          if r.level == LEVEL_GMINA and r.rodz in RODZ_AGGREGATION_SET]
n_with_est = sum(1 for r in gminas if r.est_pop.notna().any())
n_with_pop = sum(1 for r in gminas if r.pop.notna().any())
print(f"\nGminas (rodz 1/2/3): {len(gminas)}")
print(f"  with est_pop: {n_with_est} ({100*n_with_est/len(gminas):.1f}%)")
print(f"  with pop:     {n_with_pop} ({100*n_with_pop/len(gminas):.1f}%)")

# Year coverage
yr_counts = {}
for r in gminas:
    for ts in DATETIME_INDEX_FULL:
        yr = ts.year
        if pd.notna(r.est_pop.get(ts, np.nan)):
            yr_counts[yr] = yr_counts.get(yr, 0) + 1
print(f"\nest_pop coverage by year (sample):")
for yr in sorted(yr_counts)[:5]:
    print(f"  {yr}: {yr_counts[yr]} gminas")
print(f"  ...")
for yr in sorted(yr_counts)[-5:]:
    print(f"  {yr}: {yr_counts[yr]} gminas")

  ✓ Extracted est_pop for 0 records (3058 total with est_pop)

Completed in 1.7s

Gminas (rodz 1/2/3): 2660
  with est_pop: 2660 (100.0%)
  with pop:     2660 (100.0%)

est_pop coverage by year (sample):
  1986: 2660 gminas
  1987: 2660 gminas
  1988: 2660 gminas
  1989: 2660 gminas
  1990: 2660 gminas
  ...
  2021: 2477 gminas
  2022: 2477 gminas
  2023: 2477 gminas
  2024: 2477 gminas
  2025: 2477 gminas


In [3]:
# ── Cell 3: Spot-check est_pop vs pop for a sample gmina ──
sample_tid = '1261011'  # Kraków
rec = db._records.get(sample_tid)
if rec:
    print(f"Spot check: {sample_tid} ({rec.name}), rodz={rec.rodz}")
    print(f"\n{'Year':<6s}  {'pop':>10s}  {'est_pop':>10s}  {'diff%':>8s}")
    print("-" * 40)
    for yr in range(1995, 2026):
        ts = pd.Timestamp(yr, 1, 1)
        p = rec.pop.get(ts, np.nan)
        e = rec.est_pop.get(ts, np.nan)
        if pd.notna(p) and pd.notna(e):
            diff_pct = 100 * abs(e - p) / p if p > 0 else 0
            print(f"  {yr}  {p:>10.0f}  {e:>10.0f}  {diff_pct:>7.2f}%")
        elif pd.notna(e):
            print(f"  {yr}  {'—':>10s}  {e:>10.0f}")
else:
    print(f"Record {sample_tid} not found")

gc.collect()

Spot check: 1261011 (Kraków), rodz=1

Year           pop     est_pop     diff%
----------------------------------------
  1995      744987      744987     0.00%
  1996      740675      740675     0.00%
  1997      740537      740537     0.00%
  1998      740666      740666     0.00%
  1999      755355      755355     0.00%
  2000      758715      758715     0.00%
  2001      757942      757942     0.00%
  2002      757547      757547     0.00%
  2003      757685      757685     0.00%
  2004      757430      757430     0.00%
  2005      756629      756629     0.00%
  2006      756267      756267     0.00%
  2007      756583      756583     0.00%
  2008      754624      754624     0.00%
  2009      755000      755000     0.00%
  2010      757740      757740     0.00%
  2011      759137      759137     0.00%
  2012      758334      758334     0.00%
  2013      758992      758992     0.00%
  2014      761873      761873     0.00%
  2015      761069      761069     0.00%
  2016      765320 

0

## Step 2: Classify Gminas by Population Class

In [4]:
# ── Cell 4: Classify gminas using est_pop ──
t0 = time.time()
n_classified = db.classify_est_population(verbose=True)
elapsed = time.time() - t0
print(f"Completed in {elapsed:.1f}s")

# Summary
gminas_with_pc = [r for r in db._records.values()
                  if r.level == LEVEL_GMINA and r.rodz in RODZ_AGGREGATION_SET
                  and r.pop_class]
print(f"\nGminas with pop_class: {len(gminas_with_pc)}/{len(gminas)}")

# Count by rodz
for rodz in ['1', '2', '3']:
    n = sum(1 for r in gminas_with_pc if r.rodz == rodz)
    print(f"  rodz {rodz}: {n}")

# Sample distribution for year 2020
print(f"\nNational pop_class distribution at 2020:")
agg_2020 = {}
for r in gminas_with_pc:
    dist = r.pop_class.get(2020, {})
    for (code, label), val in dist.items():
        agg_2020[label] = agg_2020.get(label, 0.0) + val

total = sum(agg_2020.values())
for label in sorted(agg_2020, key=lambda x: agg_2020[x], reverse=True):
    pct = 100 * agg_2020[label] / total if total > 0 else 0
    print(f"  {label:<40s} {agg_2020[label]:>12,.0f}  ({pct:>5.1f}%)")

  ✓ Classified 2660 gminas (rodz 1/2/3) by locality type
Completed in 2.1s

Gminas with pop_class: 2660/2660
  rodz 1: 314
  rodz 2: 1623
  rodz 3: 723

National pop_class distribution at 2020:
  wieś                                       15,311,289  ( 40.2%)
  miasto od 100 001 do 500 000                6,060,472  ( 15.9%)
  miasto do 20 000                            4,946,360  ( 13.0%)
  miasto 500 001 i więcej                     4,556,677  ( 12.0%)
  miasto od 20 001 do 50 000                  4,098,465  ( 10.8%)
  miasto od 50 001 do 100 000                 3,115,301  (  8.2%)


In [5]:
# ── Cell 5: Spot-check rodz=3 gmina (urban-rural split) ──
# Find a rodz=3 gmina to inspect
rodz3 = [r for r in gminas_with_pc if r.rodz == '3']
sample_r3 = rodz3[0] if rodz3 else None
if sample_r3:
    print(f"Rodz=3 example: {sample_r3.teryt_id} ({sample_r3.name})")
    child4_tid = sample_r3.teryt_id[:-1] + '4'
    child5_tid = sample_r3.teryt_id[:-1] + '5'
    rec4 = db._records.get(child4_tid)
    rec5 = db._records.get(child5_tid)
    
    print(f"  Child rodz=4: {child4_tid} ({rec4.name if rec4 else '—'})")
    print(f"  Child rodz=5: {child5_tid} ({rec5.name if rec5 else '—'})")
    
    print(f"\n{'Year':<6s}  {'Parent est_pop':>14s}  {'Split':>40s}")
    print("-" * 65)
    for yr in [1990, 1995, 2000, 2005, 2010, 2015, 2020]:
        ts = pd.Timestamp(yr, 1, 1)
        est = sample_r3.est_pop.get(ts, np.nan)
        dist = sample_r3.pop_class.get(yr, {})
        dist_str = ', '.join(f'{lbl}: {v:.0f}' for (c, lbl), v in dist.items())
        if pd.notna(est):
            print(f"  {yr}  {est:>14.0f}  {dist_str}")
        else:
            print(f"  {yr}  {'—':>14s}  {dist_str if dist_str else '—'}")
else:
    print("No rodz=3 gminas found")

gc.collect()

Rodz=3 example: 0201043 (Nowogrodziec)
  Child rodz=4: 0201044 (Nowogrodziec)
  Child rodz=5: 0201045 (Nowogrodziec)

Year    Parent est_pop                                     Split
-----------------------------------------------------------------
  1990           14230  miasto do 20 000: 3809, wieś: 10421
  1995           14676  miasto do 20 000: 4083, wieś: 10593
  2000           14665  miasto do 20 000: 4128, wieś: 10537
  2005           14729  miasto do 20 000: 4056, wieś: 10673
  2010           15218  miasto do 20 000: 4205, wieś: 11013
  2015           15281  miasto do 20 000: 4259, wieś: 11022
  2020           14852  miasto do 20 000: 4167, wieś: 10685


0

## Step 3: Aggregate Pop-Class to Higher Levels

In [6]:
# ── Cell 6: Aggregate pop_class to powiat/voivodeship ──
t0 = time.time()
n_agg = db.aggregate_pop_class(verbose=True)
elapsed = time.time() - t0
print(f"Completed in {elapsed:.1f}s")

# Count by level
for level, lbl in [(LEVEL_POWIAT, 'Powiat'), (LEVEL_VOIVODESHIP, 'Voivodeship')]:
    n = sum(1 for r in db._records.values()
            if r.level == level and r.pop_class)
    print(f"  {lbl}: {n} records with pop_class")

  ✓ Aggregated pop_class for 400 higher-level records
Completed in 0.2s
  Powiat: 382 records with pop_class
  Voivodeship: 18 records with pop_class


In [7]:
# ── Cell 7: Show voivodeship distributions for 2020 ──
voivs = [r for r in db._records.values()
         if r.level == LEVEL_VOIVODESHIP and r.pop_class]

print(f"{'Voivodeship':<25s}  {'wieś':>12s}  {'≤20k':>10s}  {'20-50k':>10s}  "
      f"{'50-100k':>10s}  {'100-500k':>10s}  {'>500k':>10s}  {'Total':>12s}")
print("-" * 110)

CLASS_ORDER = [
    (1, 'wieś'),
    (2, 'miasto do 20 000'),
    (3, 'miasto od 20 001 do 50 000'),
    (4, 'miasto od 50 001 do 100 000'),
    (5, 'miasto od 100 001 do 500 000'),
    (6, 'miasto 500 001 i więcej'),
]

for rec in sorted(voivs, key=lambda r: r.name):
    dist = rec.pop_class.get(2020, {})
    total = sum(dist.values())
    vals = [dist.get(key, 0.0) for key in CLASS_ORDER]
    print(f"  {rec.name:<23s}  " + "  ".join(f"{v:>10,.0f}" for v in vals) + f"  {total:>12,.0f}")
    
gc.collect()

Voivodeship                        wieś        ≤20k      20-50k     50-100k    100-500k       >500k         Total
--------------------------------------------------------------------------------------------------------------
  DOLNOŚLĄSKIE                906,452     502,713     331,066     362,302     105,007     673,592     2,881,132
  KUJAWSKO-POMORSKIE          818,406     330,993      53,760     161,157     643,049           0     2,007,365
  LUBELSKIE                 1,072,041     248,731     190,220     175,618     335,114           0     2,021,724
  LUBUSKIE                    347,543     262,772     119,087           0     260,674           0       990,076
  MAZOWIECKIE               1,819,759     508,813     629,793     246,950     317,233   1,861,774     5,384,322
  MAŁOPOLSKIE               1,755,814     389,932     271,255      81,435     106,596     800,531     3,405,563
  Mazowiecki regionalny     1,201,657     320,276     267,997      76,471     317,233           0     2

0

## Step 4: JSON Export of Estimated Tables

In [8]:
in_root = os.path.join(REPO, '..', '..', 'Data', 'reweighing')

In [9]:
from local_utility_functions import remove_polish_characters

MACROREGION_OLD_YEARS = [1986, 1992]
OLD_YEARS = [1995]
NEW_YEARS = [1999, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

lis_macroregions = {1: ['5800000', '9400000', '7500000', '6900000', '6700000'],
                    2: ['9800000', '5200000', '7700000', '6400000'],
                    3: ['9200000', '9100000', '8400000', '6200000', '6300000'],
                    4: ['9700000', '9300000', '8700000', '7600000'],
                    5: ['8600000', '8300000', '8200000', '7800000', '7100000', '6800000', '6100000', '6000000'],
                    6: ['9900000', '9500000', '7900000', '5400000'],
                    7: ['5100000', '7300000', '7000000', '6600000', '6500000'],
                    8: ['9600000', '8800000', '8500000', '8000000', '7400000', '7200000', '5900000', '5500000'],
                    9: ['9000000', '8900000', '8100000', '5700000', '5600000', '5300000']
                    }

name_json_dict = f"LIS_macroregions"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

all_years_dict = {}
for year in MACROREGION_OLD_YEARS:
    year_dict = {}
    for macroregion, voiv in lis_macroregions.items():
        voiv_dict = {}
        children = []
        for voiv in lis_macroregions[macroregion]:
            children.extend(db.get_by_teryt_id(voiv).children_ids[year])
        export_multi = db.export_tables_json(
        children,
        years=year,
        subject_ids=['E_age_sex_1990',
                    'E_educ_sex_1990',
                    'E_hh_size_1990'],
        )
        voiv_dict = {str(k): v for k, v in export_multi.items()}
        year_dict[str(macroregion)] = voiv_dict
    all_years_dict[str(year)] = year_dict

json_path = os.path.join(JSON_ROOT, name_json_dict, f'LIS_all_years.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_years_dict, f, ensure_ascii=False, indent=2)
    
    
lis_old_voiv = {remove_polish_characters(str(db.get_by_teryt_id(f"{100-id}00000").name).lower().strip()): db.get_by_teryt_id(f"{100-id}00000").teryt_id for id in range(1,50)}

name_json_dict = f"LIS_old_voiv"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

all_years_dict = {}
for year in OLD_YEARS:
    year_dict = {}
    for region, voiv in lis_old_voiv.items():
        voiv_dict = {}
        children = db.get_by_teryt_id(voiv).children_ids[year]
        export_multi = db.export_tables_json(
        children,
        years=year,
        subject_ids=['E_age_sex_1990',
                    'E_educ_sex_1990',
                    'E_hh_size_1990'],
        )
        voiv_dict = {str(k): v for k, v in export_multi.items()}
        year_dict[str(region)] = voiv_dict
    all_years_dict[str(year)] = year_dict

json_path = os.path.join(JSON_ROOT, name_json_dict, f'LIS_all_years.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_years_dict, f, ensure_ascii=False, indent=2)
    
lis_new_voiv = {remove_polish_characters(str(db.get_by_teryt_id(f"{str(id).zfill(2)}00000").name).lower().strip()): db.get_by_teryt_id(f"{str(id).zfill(2)}00000").teryt_id for id in range(2,34,2)}

name_json_dict = f"LIS_new_voiv"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

all_years_dict = {}
for year in NEW_YEARS:
    year_dict = {}
    for region, voiv in lis_new_voiv.items():
        voiv_dict = {}
        export_multi = db.export_tables_json(
        voiv,
        years=year,
        subject_ids=['E_age_sex_2000',
                    'E_educ_sex_2000',
                    'E_hh_size_2000'],
        )
        voiv_dict = {str(k): v for k, v in export_multi.items()}
        year_dict[str(region)] = voiv_dict
    all_years_dict[str(year)] = year_dict
    json_path = os.path.join(JSON_ROOT, name_json_dict, f'LIS_{year}.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(year_dict, f, ensure_ascii=False, indent=2)

json_path = os.path.join(JSON_ROOT, name_json_dict, f'LIS_all_years.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_years_dict, f, ensure_ascii=False, indent=2)
    
gc.collect()

165

In [10]:
in_root = os.path.join(REPO, '..', '..', 'Data', 'reweighing')

# open the JSON file U_lis_macroregions_dict.json
file_name = 'U_lis_macroregions_dict'
with open(os.path.join(in_root, f'{file_name}.json'), 'r') as f:
    U_lis_macroregions_dict = json.load(f)

In [11]:
U_teryt_regions = {}
for key in U_lis_macroregions_dict.keys():
    U_teryt_regions[key] = U_lis_macroregions_dict[key]['array']

U_teryt_keys = {}
for key in U_lis_macroregions_dict.keys():
    U_teryt_keys[key] = U_lis_macroregions_dict[key]['keys']

LIS_OLD_YEARS = [1995]
LIS_NEW_YEARS = [1999, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
LIS_MACRO_YEARS = [1986, 1992]

U_100_200_500_groups = {}
for key in U_lis_macroregions_dict.keys():
    val = U_lis_macroregions_dict[key]['keys']
    if len(val) > 0:
        for v in val:
            if 125 == v[2] and v[1] in LIS_MACRO_YEARS:
                U_100_200_500_groups[str(tuple(v))] = key
                
name_json_dict = f"{file_name}"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

# Save meta data for U_teryt_regions
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_LIS_teryt_keys.json'), 'w') as f:
    json.dump(U_teryt_keys, f, indent=4)
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_LIS_teryt_regions.json'), 'w') as f:
    json.dump(U_teryt_regions, f, indent=4)
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_LIS_100_200_500_groups.json'), 'w') as f:
    json.dump(U_100_200_500_groups, f, indent=4)
    
all_groups_dict = {}
for g in U_teryt_regions.keys():
    export_multi = db.export_tables_json(
        U_teryt_regions[g],
        years=LIS_MACRO_YEARS,
        subject_ids=['E_age_sex_1990',
                    'E_educ_sex_1990',
                    'E_hh_size_1990'],
    )
    all_groups_dict[g] = export_multi
    json_ready = {str(k): v for k, v in export_multi.items()}
    json_path = os.path.join(JSON_ROOT, name_json_dict, f'LIS_{g}.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(json_ready, f, ensure_ascii=False, indent=2)
    
json_path = os.path.join(JSON_ROOT, name_json_dict, f'LIS_all_groups.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_groups_dict, f, ensure_ascii=False, indent=2)

del all_groups_dict    
gc.collect()
    

1023

In [12]:
# open the JSON file U_old_voiv_dict.json
file_name = 'U_old_voiv_dict'
with open(os.path.join(in_root, f'{file_name}.json'), 'r') as f:
    U_old_voiv_dict = json.load(f)

In [13]:
U_teryt_regions = {}
for key in U_old_voiv_dict.keys():
    U_teryt_regions[key] = U_old_voiv_dict[key]['array']

U_teryt_keys = {}
for key in U_old_voiv_dict.keys():
    U_teryt_keys[key] = U_old_voiv_dict[key]['keys']
    
U_100_200_500_groups = {}
for key in U_old_voiv_dict.keys():
    val = U_old_voiv_dict[key]['keys']
    if len(val) > 0:
        for v in val:
            if 125 == v[2] and v[1] in LIS_OLD_YEARS:
                U_100_200_500_groups[str(tuple(v))] = key
                
name_json_dict = f"{file_name}"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

# Save meta data for U_teryt_regions
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_LIS_100_200_500_groups.json'), 'w') as f:
    json.dump(U_100_200_500_groups, f, indent=4)
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_LIS_teryt_keys.json'), 'w') as f:
    json.dump(U_teryt_keys, f, indent=4)
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_LIS_teryt_regions.json'), 'w') as f:
    json.dump(U_teryt_regions, f, indent=4)
    
all_groups_dict = {}
for g in U_teryt_regions.keys():
    export_multi = db.export_tables_json(
        U_teryt_regions[g],
        years=LIS_OLD_YEARS,
        subject_ids=['E_age_sex_1990',
                    'E_educ_sex_1990',
                    'E_hh_size_1990'],
    )
    all_groups_dict[g] = export_multi
    json_ready = {str(k): v for k, v in export_multi.items()}
    json_path = os.path.join(JSON_ROOT, name_json_dict, f'LIS_{g}.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(json_ready, f, ensure_ascii=False, indent=2)
    
json_path = os.path.join(JSON_ROOT, name_json_dict, f'LIS_all_groups.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_groups_dict, f, ensure_ascii=False, indent=2)
    
del all_groups_dict
gc.collect()

891

In [14]:
# open the JSON file U_old_voiv_dict.json
file_name = 'U_new_voiv_dict'
with open(os.path.join(in_root, f'{file_name}.json'), 'r') as f:
    U_new_voiv_dict = json.load(f)

In [15]:
U_teryt_regions = {}
for key in U_new_voiv_dict.keys():
    U_teryt_regions[key] = U_new_voiv_dict[key]['array']

U_teryt_keys = {}
for key in U_new_voiv_dict.keys():
    to_filter = U_new_voiv_dict[key]['keys']
    for_key = []
    for row in to_filter:
        if 125 == row[2] and row[1] in LIS_NEW_YEARS:
            for_key.append(row)
    U_teryt_keys[key] = for_key
    
U_100_200_500_groups = {}
for key in U_new_voiv_dict.keys():
    val = U_new_voiv_dict[key]['keys']
    if len(val) > 0:
        for v in val:
            if 125 == v[2] and v[1] in LIS_NEW_YEARS:
                U_100_200_500_groups[str(tuple(v))] = key
                
name_json_dict = f"{file_name}"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

# Save meta data for U_teryt_regions
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_LIS_100_200_500_groups.json'), 'w') as f:
    json.dump(U_100_200_500_groups, f, indent=4)
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_LIS_teryt_keys.json'), 'w') as f:
    json.dump(U_teryt_keys, f, indent=4)
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_LIS_teryt_regions.json'), 'w') as f:
    json.dump(U_teryt_regions, f, indent=4)
    
all_groups_dict = {}
for g in U_teryt_regions.keys():
    export_multi = db.export_tables_json(
        U_teryt_regions[g],
        years=LIS_NEW_YEARS,
        subject_ids=['E_age_sex_1990', 'E_age_sex_2000',
                    'E_educ_sex_1990', 'E_educ_sex_2000',
                    'E_hh_size_1990', 'E_hh_size_2000'],
    )
    all_groups_dict[g] = export_multi
    json_ready = {str(k): v for k, v in export_multi.items()}
    json_path = os.path.join(JSON_ROOT, name_json_dict, f'LIS_{g}.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(json_ready, f, ensure_ascii=False, indent=2)
    
json_path = os.path.join(JSON_ROOT, name_json_dict, f'LIS_all_groups.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_groups_dict, f, ensure_ascii=False, indent=2)
    
del all_groups_dict 
gc.collect()

396

In [16]:
parent_json_dict = f"{file_name}_PER_YEAR"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

for year in LIS_NEW_YEARS:
    U_teryt_regions = {}
    for key in U_new_voiv_dict.keys():
        U_teryt_regions[key] = U_new_voiv_dict[key]['array']

    U_teryt_keys = {}
    for key in U_new_voiv_dict.keys():
        to_filter = U_new_voiv_dict[key]['keys']
        for_key = []
        for row in to_filter:
            if 125 == row[2] and row[1] == year:
                for_key.append(row)
        U_teryt_keys[key] = for_key

    U_100_200_500_groups = {}
    for key in U_new_voiv_dict.keys():
        val = U_new_voiv_dict[key]['keys']
        if len(val) > 0:
            for v in val:
                if 125 == v[2] and v[1] == year:
                    U_100_200_500_groups[str(tuple(v))] = key
    
    name_json_dict = f"{file_name}_{year}"
    # Make folder in JSON_ROOT if it doesn't exist
    os.makedirs(os.path.join(JSON_ROOT, parent_json_dict, name_json_dict), exist_ok=True)
    
    
    # Save meta data for U_teryt_regions
    with open(os.path.join(JSON_ROOT, parent_json_dict, name_json_dict, 'META_LIS_100_200_500_groups.json'), 'w') as f:
        json.dump(U_100_200_500_groups, f, indent=4)
    with open(os.path.join(JSON_ROOT, parent_json_dict, name_json_dict, 'META_LIS_teryt_keys.json'), 'w') as f:
        json.dump(U_teryt_keys, f, indent=4)
    with open(os.path.join(JSON_ROOT, parent_json_dict, name_json_dict, 'META_LIS_teryt_regions.json'), 'w') as f:
        json.dump(U_teryt_regions, f, indent=4)
        
    all_groups_dict = {}
    for g in U_teryt_regions.keys():
        export_multi = db.export_tables_json(
            U_teryt_regions[g],
            years=year,
            subject_ids=['E_age_sex_1990', 'E_age_sex_2000',
                        'E_educ_sex_1990', 'E_educ_sex_2000',
                        'E_hh_size_1990', 'E_hh_size_2000'],
        )
        all_groups_dict[g] = {str(k): v for k, v in export_multi.items()}
        
    json_path = os.path.join(JSON_ROOT, parent_json_dict, name_json_dict, f'LIS_all_groups.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(all_groups_dict, f, ensure_ascii=False, indent=2)
    
    all_groups_dict = {}
    for g in list(set(list(U_100_200_500_groups.values()))):
        export_multi = db.export_tables_json(
            U_teryt_regions[g],
            years=year,
            subject_ids=['E_age_sex_1990', 'E_age_sex_2000',
                        'E_educ_sex_1990', 'E_educ_sex_2000',
                        'E_hh_size_1990', 'E_hh_size_2000'],
        )
        all_groups_dict[g] = export_multi
        json_ready = {str(k): v for k, v in export_multi.items()}
    
    json_path = os.path.join(JSON_ROOT, parent_json_dict, name_json_dict, f'LIS_all_relevant_groups.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(all_groups_dict, f, ensure_ascii=False, indent=2)
    
del all_groups_dict
gc.collect()

66

In [17]:
# Only voivodeships

name_json_dict = f"CBOS_old_voiv"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

YEARS = [1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000]

cbos_old_voiv = {str(db.get_by_teryt_id(f"{100-id}00000").name).lower().strip(): db.get_by_teryt_id(f"{100-id}00000").teryt_id for id in range(1,50)}

all_years_dict = {}
for year in YEARS:
    year_dict = {}
    for region, voiv in cbos_old_voiv.items():
        voiv_dict = {}
        children = db.get_by_teryt_id(voiv).children_ids[year]
        export_multi = db.export_tables_json(
        children,
        years=year,
        subject_ids=['E_age_sex_1990',
                    'E_educ_sex_1990',
                    'E_hh_size_1990'],
        )
        voiv_dict = {str(k): v for k, v in export_multi.items()}
        year_dict[str(region)] = voiv_dict
    all_years_dict[str(year)] = year_dict

json_path = os.path.join(JSON_ROOT, name_json_dict, f'CBOS_all_years.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_years_dict, f, ensure_ascii=False, indent=2)
    
cbos_new_voiv = {str(db.get_by_teryt_id(f"{str(id).zfill(2)}00000").name).lower().strip(): db.get_by_teryt_id(f"{str(id).zfill(2)}00000").teryt_id for id in range(2,34,2)}

name_json_dict = f"CBOS_new_voiv"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

YEARS = [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]

all_years_dict = {}
for year in YEARS:
    year_dict = {}
    for region, voiv in cbos_new_voiv.items():
        voiv_dict = {}
        export_multi = db.export_tables_json(
        voiv,
        years=year,
        subject_ids=['E_age_sex_2000',
                    'E_educ_sex_2000',
                    'E_hh_size_2000'],
        )
        voiv_dict = {str(k): v for k, v in export_multi.items()}
        year_dict[str(region)] = voiv_dict
    all_years_dict[str(year)] = year_dict

json_path = os.path.join(JSON_ROOT, name_json_dict, f'CBOS_all_years.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_years_dict, f, ensure_ascii=False, indent=2)
    

name_json_dict = f"CBOS_macroregions"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

YEARS = [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]

cbos_macroregions = {1: ['1400000', '1000000'],
                     2: ['1200000', '2400000'],
                     3: ['0600000', '1800000', '2000000', '2600000'],
                     4: ['3000000', '3200000', '0800000'],
                     5: ['0200000', '1600000'],
                     6: ['2200000', '2800000', '0400000']
                     }

all_years_dict = {}
for year in YEARS:
    year_dict = {}
    for macroregion, voiv in cbos_macroregions.items():
        voiv_dict = {}
        children = []
        for voiv in cbos_macroregions[macroregion]:
            children.extend(db.get_by_teryt_id(voiv).children_ids[year])
        export_multi = db.export_tables_json(
        children,
        years=year,
        subject_ids=['E_age_sex_2000',
                    'E_educ_sex_2000',
                    'E_hh_size_2000'],
        )
        voiv_dict = {str(k): v for k, v in export_multi.items()}
        year_dict[str(macroregion)] = voiv_dict
    all_years_dict[str(year)] = year_dict

json_path = os.path.join(JSON_ROOT, name_json_dict, f'CBOS_all_years.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_years_dict, f, ensure_ascii=False, indent=2)

gc.collect()

99

In [18]:
# open the JSON file U_old_voiv_dict.json
file_name = 'U_new_voiv_dict'
with open(os.path.join(in_root, f'{file_name}.json'), 'r') as f:
    U_new_voiv_dict = json.load(f)
parent_json_dict = f"CBOS_{file_name}"
YEARS = [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]

U_teryt_regions = {}
for key in U_new_voiv_dict.keys():
    U_teryt_regions[key] = U_new_voiv_dict[key]['array']
    
U_teryt_keys = {}
for key in U_new_voiv_dict.keys():
    to_filter = U_new_voiv_dict[key]['keys']
    for_key = []
    for row in to_filter:
        if row[2] != 125 and row[1] in YEARS:
            for_key.append(row)
    U_teryt_keys[key] = for_key
    
    
name_json_dict = f"CBOS_{file_name}"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

# Save meta data for U_teryt_regions
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_CBOS_teryt_keys.json'), 'w') as f:
    json.dump(U_teryt_keys, f, indent=4)
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_CBOS_teryt_regions.json'), 'w') as f:
    json.dump(U_teryt_regions, f, indent=4)
    
all_groups_dict = {}
for g in U_teryt_regions.keys():
    export_multi = db.export_tables_json(
        U_teryt_regions[g],
        years=YEARS,
        subject_ids=['E_age_sex_1990', 'E_age_sex_2000',
                    'E_educ_sex_1990', 'E_educ_sex_2000',
                    'E_hh_size_1990', 'E_hh_size_2000'],
    )
    all_groups_dict[g] = {str(k): v for k, v in export_multi.items()}
    
json_path = os.path.join(JSON_ROOT, name_json_dict, f'CBOS_all_groups.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_groups_dict, f, ensure_ascii=False, indent=2)
    
del all_groups_dict 
gc.collect()


33

In [19]:
# open the JSON file U_old_voiv_dict.json
file_name = 'U_old_voiv_dict'
with open(os.path.join(in_root, f'{file_name}.json'), 'r') as f:
    U_new_voiv_dict = json.load(f)
parent_json_dict = f"CBOS_{file_name}"
YEARS = [1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000]

U_teryt_regions = {}
for key in U_new_voiv_dict.keys():
    U_teryt_regions[key] = U_new_voiv_dict[key]['array']
    
U_teryt_keys = {}
for key in U_new_voiv_dict.keys():
    to_filter = U_new_voiv_dict[key]['keys']
    for_key = []
    for row in to_filter:
        if row[2] != 125 and row[1] in YEARS:
            for_key.append(row)
    U_teryt_keys[key] = for_key
    
    
name_json_dict = f"CBOS_{file_name}"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

# Save meta data for U_teryt_regions
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_CBOS_teryt_keys.json'), 'w') as f:
    json.dump(U_teryt_keys, f, indent=4)
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_CBOS_teryt_regions.json'), 'w') as f:
    json.dump(U_teryt_regions, f, indent=4)
    
all_groups_dict = {}
for g in U_teryt_regions.keys():
    export_multi = db.export_tables_json(
        U_teryt_regions[g],
        years=YEARS,
        subject_ids=['E_age_sex_1990',
                    'E_educ_sex_1990',
                    'E_hh_size_1990'],
    )
    all_groups_dict[g] = {str(k): v for k, v in export_multi.items()}
    
json_path = os.path.join(JSON_ROOT, name_json_dict, f'CBOS_all_groups.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_groups_dict, f, ensure_ascii=False, indent=2)
    
del all_groups_dict 
gc.collect()


33

In [ ]:
# open the JSON file U_old_voiv_dict.json
file_name = 'U_cbos_macroregions_dict'
with open(os.path.join(in_root, f'{file_name}.json'), 'r') as f:
    U_new_voiv_dict = json.load(f)
parent_json_dict = f"CBOS_{file_name}"
YEARS = [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]

U_teryt_regions = {}
for key in U_new_voiv_dict.keys():
    U_teryt_regions[key] = U_new_voiv_dict[key]['array']
    
U_teryt_keys = {}
for key in U_new_voiv_dict.keys():
    to_filter = U_new_voiv_dict[key]['keys']
    for_key = []
    for row in to_filter:
        if row[2] != 125 and row[1] in YEARS:
            for_key.append(row)
    U_teryt_keys[key] = for_key
    
    
name_json_dict = f"CBOS_{file_name}"
# Make folder in JSON_ROOT if it doesn't exist
os.makedirs(os.path.join(JSON_ROOT, name_json_dict), exist_ok=True)

# Save meta data for U_teryt_regions
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_CBOS_teryt_keys.json'), 'w') as f:
    json.dump(U_teryt_keys, f, indent=4)
with open(os.path.join(JSON_ROOT, name_json_dict, 'META_CBOS_teryt_regions.json'), 'w') as f:
    json.dump(U_teryt_regions, f, indent=4)
    
all_groups_dict = {}
for g in U_teryt_regions.keys():
    export_multi = db.export_tables_json(
        U_teryt_regions[g],
        years=YEARS,
        subject_ids=['E_age_sex_1990',
                    'E_educ_sex_1990',
                    'E_hh_size_1990'],
    )
    all_groups_dict[g] = {str(k): v for k, v in export_multi.items()}
    
json_path = os.path.join(JSON_ROOT, name_json_dict, f'CBOS_all_groups.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_groups_dict, f, ensure_ascii=False, indent=2)
    
del all_groups_dict 
gc.collect()


In [ ]:
# ── Cell 8: Export tables for a single gmina, single year ──
export_single = db.export_tables_json('1261011', years=2020)
print(f"Export for Kraków (1261011), year 2020:")
print(f"  Subjects: {list(export_single.get(2020, {}).keys())}")

# Show pop_class
pc = export_single.get(2020, {}).get('pop_class', {})
print(f"  pop_class: {pc}")

# Show first few entries of one subject
for sid in sorted(export_single.get(2020, {}).keys()):
    if sid == 'pop_class':
        continue
    entries = export_single[2020][sid]
    print(f"\n  {sid} ({len(entries)} cells):")
    for k, v in list(entries.items())[:6]:
        print(f"    {k}: {v}")
    if len(entries) > 6:
        print(f"    ... ({len(entries) - 6} more)")
    break

In [ ]:
# ── Cell 9: Export & aggregate multiple gminas (powiat-level) ──
# Get all gmina children of a powiat, aggregate tables
sample_powiat = '1201000'  # powiat krakowski
prec = db._records.get(sample_powiat)
if prec:
    children = prec.get_children(2020)
    gmina_children = [c for c in children
                      if db._records.get(c) and db._records[c].rodz in RODZ_AGGREGATION_SET]
    print(f"Powiat: {sample_powiat} ({prec.name}), {len(gmina_children)} gminas")
    
    export_agg = db.export_tables_json(gmina_children, years=2020)
    yr_data = export_agg.get(2020, {})
    print(f"  Aggregated subjects: {sorted(yr_data.keys())}")
    
    # Show pop_class
    pc = yr_data.get('pop_class', {})
    print(f"\n  Aggregated pop_class:")
    for label, val in sorted(pc.items(), key=lambda x: -x[1]):
        print(f"    {label:<40s} {val:>10,.0f}")
else:
    print(f"Powiat {sample_powiat} not found")

In [ ]:
# ── Cell 10: Multi-year export (JSON file) ──
# Export Kraków for all years and save to JSON
export_multi = db.export_tables_json(
    ['1200000'],
    years=list(range(1986, 2025)),
    subject_ids=['E_hh_size_1990'],
)


# Convert int keys to strings for JSON
json_ready = {str(k): v for k, v in export_multi.items()}
json_path = os.path.join(JSON_ROOT, 'example_export_krakow.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(json_ready, f, ensure_ascii=False, indent=2)
print(f"Saved export to: {json_path}")
print(f"  Years exported: {sorted(export_multi.keys())}")
print(f"  File size: {os.path.getsize(json_path) / 1024:.1f} KB")


## Step 5: Save Updated Database

In [ ]:
# ── Cell 11: Save database with est_pop and pop_class ──
print(f"Saving database to: {DB_OUTPUT}")
t0 = time.time()
db.save_complete(DB_OUTPUT, verbose=True)
elapsed = time.time() - t0
print(f"\nSaved in {elapsed:.1f}s")

In [ ]:
# Verify
print(f"\nVerification: loading saved database...")
t0 = time.time()
db_verify = load_complete_database(DB_OUTPUT, verbose=False)
elapsed = time.time() - t0

# Check est_pop
n_est = sum(1 for r in db_verify._records.values() if r.est_pop.notna().any())
# Check pop_class
n_pc = sum(1 for r in db_verify._records.values() if r.pop_class)
# Spot check
sample = db_verify._records.get('1261011')
print(f"  Loaded in {elapsed:.1f}s")
print(f"  Records with est_pop: {n_est}")
print(f"  Records with pop_class: {n_pc}")
if sample:
    print(f"  Spot check Kraków est_pop@2020: {sample.est_pop.get(pd.Timestamp(2020,1,1), 'N/A'):.0f}")
    print(f"  Spot check Kraków pop_class@2020: {sample.pop_class.get(2020, {})}")
print(f"\n✓ Database saved and verified successfully.")
del db_verify

## Summary

This notebook has:
1. ✓ Extracted `est_pop` from E_age_sex cross tables for all gminas
2. ✓ Classified all gminas (rodz 1/2/3) into population classes using `est_pop`
   - rodz 1/2: direct classification by total population
   - rodz 3: split via children (rodz 4: town, rodz 5: rural) with imputation fallback
3. ✓ Aggregated pop_class distributions to powiat, new voivodeship, and old voivodeship levels
4. ✓ Demonstrated JSON export of estimated tables (single/multiple gminas, single/multiple years)
5. ✓ Saved the updated database with `est_pop` and `pop_class` attributes

**Next:** Use the exported data for inequality analysis at the regional level.